# Day 5 — Context engineering & prompt caching
**Theory:** [day-05-context-engineering-and-caching.md](../theory/day-05-context-engineering-and-caching.md) · ⏱ ~60 min · Level: medium

**You will:**
1. Measure a multi-turn conversation **without** and **with** caching
2. Break the cache on purpose with a timestamp ("silent cache killer")
3. Trim old tool results (context hygiene), with no API needed
4. Run a mini "lost in the middle" experiment

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone + install the course repo, read the key from notebook secrets
    if not pathlib.Path("agentic-ai-course").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Rahulverma5/agentic-ai-course"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "agentic-ai-course/requirements.txt"], check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("ANTHROPIC_API_KEY", userdata.get("ANTHROPIC_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("ANTHROPIC_API_KEY", UserSecretsClient().get_secret("ANTHROPIC_API_KEY"))
    except Exception as e:
        print("Add ANTHROPIC_API_KEY to the notebook secrets:", e)
    REPO = pathlib.Path("agentic-ai-course").resolve()
else:  # Codespaces / local: find the repo root
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of
from shopease.data import setup_data
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

## 1. A realistic, long, *stable* system prompt
Caching only kicks in above a minimum prefix length, so we build a big prompt (policies + a product catalogue).

In [ ]:
import time, datetime, json
import pandas as pd
policies = "\n".join(p.read_text() for p in sorted(pathlib.Path("projects/shopease-support-agent/data/policies").glob("*.md")))
catalogue = "\n".join(f"- SKU{i:04}: Product {i}, ${(i * 7) % 200 + 5}.00, category {['home', 'apparel', 'electronics'][i % 3]}"
                      for i in range(600))
SYSTEM = f"## Role\nYou are ShopEase support.\n\n## Policies\n{policies}\n\n## Catalogue\n{catalogue}"
print(client.messages.count_tokens(model=MODEL, system=SYSTEM,
                                   messages=[{"role": "user", "content": "hi"}]).input_tokens, "tokens in the prefix")

QUESTIONS = ["What's the return window?", "And for opened earbuds?", "How much is SKU0042?",
             "Do gold members pay return shipping?", "Summarise our chat in one line."]

def conversation(cache: bool, system_fn=lambda: SYSTEM, model=CHEAP_MODEL):
    msgs, rows = [], []
    for q in QUESTIONS:
        msgs.append({"role": "user", "content": q})
        kw = {"cache_control": {"type": "ephemeral"}} if cache else {}
        t0 = time.time()
        r = client.messages.create(model=model, max_tokens=300, system=system_fn(), messages=msgs, **kw)
        cost = tracker.add(model, r.usage, f"cache={cache}")
        msgs.append({"role": "assistant", "content": r.content})
        rows.append({"turn": len(rows) + 1, "input": r.usage.input_tokens,
                     "cache_write": r.usage.cache_creation_input_tokens or 0,
                     "cache_read": r.usage.cache_read_input_tokens or 0,
                     "seconds": round(time.time() - t0, 2), "cost $": round(cost, 6)})
    return pd.DataFrame(rows)

## 2. Without vs with caching

In [ ]:
no_cache = conversation(cache=False)
with_cache = conversation(cache=True)
print("WITHOUT cache"); display(no_cache)
print("WITH cache"); display(with_cache)
print(f"total: ${no_cache['cost $'].sum():.5f} vs ${with_cache['cost $'].sum():.5f}")

> If `cache_read` stays 0, the prefix may be below the model's minimum cacheable length: make the catalogue longer.

**Savings math:** saving ≈ 1 − (w + (n−1)·r)/n, with write multiplier w≈1.25 and read multiplier r.

In [ ]:
for r_mult in (0.05, 0.1):
    for n in (5, 10, 20):
        print(f"r={r_mult}, n={n:>2}: saving on prefix ≈ {1 - (1.25 + (n - 1) * r_mult) / n:.0%}")

## 3. The silent cache killer
Putting `datetime.now()` at the top of the system prompt changes the prefix on every call.

In [ ]:
killer = conversation(cache=True, system_fn=lambda: f"Current time: {datetime.datetime.now().isoformat()}\n\n{SYSTEM}")
display(killer[["turn", "cache_write", "cache_read", "cost $"]])
print("Fix: put volatile data (time, user id) at the END of the prompt, after the cached prefix.")

## 4. Context hygiene: trim old tool results (no API needed)

In [ ]:
def trim_old_tool_results(messages, keep_last=2, placeholder="[old tool result removed]"):
    seen = 0
    for msg in reversed(messages):
        if msg["role"] == "user" and isinstance(msg["content"], list):
            for block in msg["content"]:
                if isinstance(block, dict) and block.get("type") == "tool_result":
                    seen += 1
                    if seen > keep_last:
                        block["content"] = placeholder
    return messages

fake = []
for i in range(5):
    fake += [{"role": "assistant", "content": f"(tool call {i})"},
             {"role": "user", "content": [{"type": "tool_result", "tool_use_id": f"t{i}", "content": "x" * 5000}]}]
before = len(json.dumps(fake))
after = len(json.dumps(trim_old_tool_results(fake)))
print(f"history size: {before:,} → {after:,} characters")

## 5. Mini "lost in the middle" experiment
Hide one fact at the start, middle or end of ~15k tokens of filler and ask about it.
(Modern models are much better at this than 2023 models, so you may see little difference. That is a finding too.)

In [ ]:
filler = [f"Note {i}: the warehouse in zone {i % 9} restocked {i * 3} boxes of item {i % 50}." for i in range(900)]
FACT = "IMPORTANT: The secret discount code for Asha is MANGO-73."
def needle_test(position):
    lines = filler.copy()
    idx = {"start": 0, "middle": len(lines) // 2, "end": len(lines)}[position]
    lines.insert(idx, FACT)
    out = ask("\n".join(lines) + "\n\nWhat is Asha's secret discount code? Reply with the code only.", max_tokens=20)
    return "MANGO-73" in out

pd.DataFrame([{"position": p, "correct (3 tries)": sum(needle_test(p) for _ in range(3))} for p in ("start", "middle", "end")])

## ✏️ Exercises
1. Run `conversation(cache=True, model=MODEL)` on Opus. How do the dollar savings compare with Haiku?
2. Move the timestamp to the *end* of the system prompt (after a cache breakpoint). Does caching work again?
3. Put the fact in the middle of 60k tokens instead of 15k. Any change?

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker`. Make a habit of checking cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None